# PyTorch 常见错误调试练习

每题都模拟一个真实错误。先阅读错误原因，再修改 TODO 区域，使自动检查通过。养成优先检查 shape、dtype、device、梯度和模型模式的习惯。

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)

## 练习 1：线性层形状不匹配 ⭐

`Linear(10,4)` 要求最后一维是 10。错误写法是 `layer(batch.T)`。请给出正确输入并得到 `(32,4)` 的输出。

In [ ]:
batch = torch.randn(32, 10)
layer = nn.Linear(10, 4)

# 错误示例：output = layer(batch.T)
# TODO
output = None

assert output.shape == (32, 4)
print("✅ 练习 1 通过")

## 练习 2：CrossEntropyLoss 标签格式错误 ⭐⭐

原始标签是形状 `(4,1)` 的浮点 Tensor。请转换成 CrossEntropyLoss 需要的形状和类型。

In [ ]:
logits = torch.randn(4, 3)
raw_labels = torch.tensor([[0.0], [2.0], [1.0], [0.0]])

# TODO
labels = None
loss = None

assert labels.shape == (4,)
assert labels.dtype == torch.int64
assert torch.isfinite(loss)
print("✅ 练习 2 通过")

## 练习 3：模型与数据不在同一设备 ⭐⭐

选择 CUDA（如果可用）或 CPU，把模型、输入和标签都放到同一设备，然后完成前向和损失计算。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Linear(5, 2)
inputs = torch.randn(8, 5)
targets = torch.randn(8, 2)

# TODO
model = None
inputs = None
targets = None
prediction = None
loss = None

assert next(model.parameters()).device == device
assert inputs.device == device and targets.device == device
assert prediction.device == device and loss.device == device
print("✅ 练习 3 通过")

## 练习 4：detach 导致梯度断开 ⭐⭐

如果在计算损失前对预测调用 detach，模型参数无法获得梯度。请重新计算保持计算图的预测和损失，再反向传播。

In [ ]:
model = nn.Linear(3, 1)
inputs = torch.randn(5, 3)
targets = torch.randn(5, 1)

# 错误示例：broken_prediction = model(inputs).detach()
# TODO
prediction = None
loss = None
# 反向传播

assert loss.requires_grad
assert model.weight.grad is not None
assert model.weight.grad.abs().sum() > 0
print("✅ 练习 4 通过")

## 练习 5：忘记清空梯度 ⭐⭐⭐

补全训练循环，把 `zero_grad()` 放在正确位置。训练目标为 $y=4x-1$，最终参数应接近真实值。

In [ ]:
x = torch.linspace(-1, 1, 80).unsqueeze(1)
y = 4 * x - 1
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# TODO：训练 100 个 epoch，每轮按正确顺序清空梯度

assert abs(model.weight.item() - 4.0) < 0.1
assert abs(model.bias.item() + 1.0) < 0.1
print("✅ 练习 5 通过")

## 练习 6：评估时忘记 eval 和 no_grad ⭐⭐

模型包含 Dropout。补全 `safe_predict`，确保多次推理结果一致且不记录梯度。

In [ ]:
dropout_model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Dropout(0.7), nn.Linear(8, 2))
inputs = torch.randn(16, 4)


def safe_predict(model, inputs):
    # TODO
    pass


output_1 = safe_predict(dropout_model, inputs)
output_2 = safe_predict(dropout_model, inputs)
assert torch.equal(output_1, output_2)
assert not output_1.requires_grad
assert not dropout_model.training
print("✅ 练习 6 通过")

## 练习 7：数值不稳定的概率计算 ⭐⭐⭐

很大的 logits 可能让手写 `exp/sum(exp)` 溢出。请使用数值稳定的 PyTorch 函数计算交叉熵和 log 概率。

In [ ]:
large_logits = torch.tensor([[1000.0, 1001.0, 999.0], [-1000.0, -999.0, -1001.0]])
targets = torch.tensor([1, 1])

# TODO
stable_loss = None
stable_log_probabilities = None

assert torch.isfinite(stable_loss)
assert torch.isfinite(stable_log_probabilities).all()
assert torch.allclose(stable_log_probabilities.exp().sum(dim=1), torch.ones(2))
print("✅ 练习 7 通过")

## 调试顺序

遇到错误时依次打印：`type → shape → dtype → device → requires_grad/grad → model.training`。先定位哪条假设不成立，再修改代码。